In [1]:
import json
from pathlib import Path
import pandas as pd


In [2]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "files"

# Load annotated CVs
with open(DATA_PATH / "linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)
len(cvs)

609

In [3]:
jobs = [job for cv in cvs for job in cv]
df = pd.DataFrame(jobs)

# Focus on ACTIVE as target
df_active = df[df["status"] == "ACTIVE"].copy()

def norm_text(s: str) -> str:
    if pd.isna(s):
        return ""
    return str(s).strip().lower()

df_active["position_norm"] = df_active["position"].apply(norm_text)

In [ ]:
df_sen_v2 = pd.read_csv(DATA_PATH / "seniority-v2.csv")
df_sen_v2["text_norm"] = df_sen_v2["text"].apply(norm_text)

lookup_sen = (
    df_sen_v2.groupby("text_norm")["label"]
    .agg(lambda x: x.value_counts().idxmax())
    .to_dict()
)

In [ ]:
df_active["seniority_pred_lookup"] = df_active["position_norm"].map(lookup_sen)

match_mask = df_active["seniority_pred_lookup"].notna()
match_rate = match_mask.mean()


v2_labels = set(df_sen_v2["label"].unique())
eval_mask = match_mask & df_active["seniority"].isin(v2_labels)

accuracy_on_matched = (df_active.loc[eval_mask, "seniority_pred_lookup"]
                       == df_active.loc[eval_mask, "seniority"]).mean()

print("Seniority lookup baseline")
print("Match rate (titles found in seniority-v2):", round(match_rate, 3))
print("Accuracy on matched (truth within v2 label set):", round(accuracy_on_matched, 3))
print("Coverage for evaluation (matched & truth in v2):", int(eval_mask.sum()), "/", len(df_active))

Seniority lookup baseline
Match rate (titles found in seniority-v2): 0.228
Accuracy on matched (truth within v2 label set): 0.793
Coverage for evaluation (matched & truth in v2): 121 / 623
